In [4]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import (
    r2_score,
    mean_absolute_error,
    mean_squared_error
)

df = pd.read_csv("cars.csv")

df.head()

,brand,km_driven,fuel,owner,selling_price
0,Maruti,145500,Diesel,First Owner,450000
1,Skoda,120000,Diesel,Second Owner,370000
2,Honda,140000,Petrol,Third Owner,158000
3,Hyundai,127000,Diesel,First Owner,225000
4,Maruti,120000,Petrol,First Owner,130000


In [5]:
df.shape

(8128, 5)

In [6]:
brand_counts = df['brand'].value_counts()

df['brand'] = df['brand'].where(
    df['brand'].map(brand_counts) >= 5,
    'Other'
)

print(df['brand'].value_counts())

brand
Maruti           2448
Hyundai          1415
Mahindra          772
Tata              734
Toyota            488
Honda             467
Ford              397
Chevrolet         230
Renault           228
Volkswagen        186
BMW               120
Skoda             105
Nissan             81
Jaguar             71
Volvo              67
Datsun             65
Mercedes-Benz      54
Fiat               47
Audi               40
Lexus              34
Jeep               31
Other              17
Mitsubishi         14
Land                6
Force               6
Isuzu               5
Name: count, dtype: int64


In [7]:
X = df.drop('selling_price', axis=1)

y = df['selling_price']

print("X shape:", X.shape)
print("y shape:", y.shape)

X shape: (8128, 4)
y shape: (8128,)


In [8]:
categorical_cols = ['brand', 'fuel', 'owner']
numerical_cols = ['km_driven']

In [9]:
preprocessor = ColumnTransformer(
    transformers=[
        (
            'cat',
            OneHotEncoder(
                handle_unknown='ignore',
                min_frequency=2
            ),
            categorical_cols
        ),
        (
            'num',
            SimpleImputer(strategy='median'),
            numerical_cols
        )
    ]
)

In [10]:
model = RandomForestRegressor(
    n_estimators=500,
    max_depth=None,
    min_samples_split=2,
    min_samples_leaf=2,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1
)

In [11]:
pipe = Pipeline([
    ('preprocessor', preprocessor),
    ('model', model)
])

In [12]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [13]:
print("Training data:", X_train.shape)
print("Testing data:", X_test.shape)

Training data: (6502, 4)
Testing data: (1626, 4)


In [14]:
pipe.fit(X_train, y_train)

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(transformers=[('cat',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                min_frequency=2),
                                                  ['brand', 'fuel', 'owner']),
                                                 ('num',
                                                  SimpleImputer(strategy='median'),
                                                  ['km_driven'])])),
                ('model',
                 RandomForestRegressor(max_features='sqrt', min_samples_leaf=2,
                                       n_estimators=500, n_jobs=-1,
                                       random_state=42))])

In [15]:
y_pred = pipe.predict(X_test)

print(y_pred[:10])

[295128.12948799 212131.38927685 618358.29901605 279833.8273677
 536581.40373349 182733.26465732 552490.90579428 582385.67729398
 157138.87018718 345888.35590432]


In [16]:
r2 = r2_score(y_test, y_pred)

print("R² Score:", r2)

R² Score: 0.8607164006174222


In [17]:
print("R² Score: {:.2f}%".format(r2 * 100))

R² Score: 86.07%


In [18]:
mae = mean_absolute_error(y_test, y_pred)

print("MAE:", mae)

MAE: 175459.69116805773


In [19]:
rmse = np.sqrt(mean_squared_error(y_test, y_pred))

print("RMSE:", rmse)

RMSE: 302155.97329663584


In [20]:
mape = np.mean(
    np.abs((y_test - y_pred) / y_test)
) * 100

print("MAPE: {:.2f}%".format(mape))

MAPE: 45.63%


In [21]:
print("========== MODEL PERFORMANCE ==========")

print("R² Score  :", round(r2, 4))
print("R² Percent:", round(r2 * 100, 2), "%")

print("MAE       :", round(mae, 2))
print("RMSE      :", round(rmse, 2))
print("MAPE      :", round(mape, 2), "%")

========== MODEL PERFORMANCE ==========
R² Score  : 0.8607
R² Percent: 86.07 %
MAE       : 175459.69
RMSE      : 302155.97
MAPE      : 45.63 %
